# 03 - Trait space in 3-D: Fig. 2a

**Purpose.** Draw the three-dimensional functional trait space (PC1, PC2, PC3). The 10,000-point
sample is projected onto the published PCA axes; points are coloured by NLCD land-cover class and
arrows show the trait loadings.

**Inputs.** `../data/pca_model/pca_sample_points.csv`, `../data/pca_model/scaler_rp_10traits.pkl`,
`../data/pca_model/PCA_model_rp_sa_10traits.pkl` (the model fitted in notebook 01).

**Outputs.** `./results/fig2a_trait_space_3d.png` (Fig. 2a).

Run with the working directory set to this folder.

In [ ]:
import os
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
from matplotlib import cm

DATA_DIR = '../data/pca_model'
OUT_DIR = './results'
os.makedirs(OUT_DIR, exist_ok=True)

# the ten traits entering the PCA (column names after renaming in the next cell)
trait_list = ['Carbon', 'Cellulose', 'Chlorophyll a + b', 'EWT', 'Lignin', 'Nitrogen',
              'NSC', 'Phenolics', 'SLA', 'Canopy Height']

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'pca_sample_points.csv'))
df.rename(columns={'ChlorophyllsArea': 'Chlorophyll a + b', 'canopy_height': 'Canopy Height'}, inplace=True)
# canopy height is an integer raster value; +1 avoids log(0) for zero-height pixels
df['Canopy Height'] = df['Canopy Height'] + 1

In [ ]:
df = df.dropna()

In [ ]:
# natural vegetation only: drop NLCD 81 (pasture/hay) and 82 (cultivated crops)
df = df.query('nlcd_class != 81 and nlcd_class != 82')

In [ ]:
# random subsample of 10,000 points (fixed seed) used for the PCA and the trait-space figures
df = df.sample(n=10000, random_state=1)

In [ ]:
nlcd_dict = {0: 'Open Water', 11: 'Developed, Open Space', 12: 'Developed, Low Intensity',
             21: 'Developed, Medium Intensity',
             22: 'Developed, High Intensity', 23: 'Developed, Open Space with Buildings',
             24: 'Developed, Open Space with Roads',
             31: 'Barren Land (Rock/Sand/Clay)', 41: 'Deciduous Forest', 42: 'Evergreen Forest', 43: 'Mixed Forest',
             51: 'Dwarf Scrub', 52: 'Shrub/Scrub', 71: 'Grassland/Herbaceous', 72: 'Sedge/Herbaceous', 73: 'Lichens',
             74: 'Moss', 81: 'Pasture/Hay', 82: 'Cultivated Crops', 90: 'Woody Wetlands',
             95: 'Emergent Herbaceous Wetlands'}
df['nlcd'] = df['nlcd_class'].map(nlcd_dict)

# Load the published scaler and PCA (fitted in notebook 01)

In [ ]:
scaler_all = pickle.load(open(os.path.join(DATA_DIR, 'scaler_rp_10traits.pkl'), 'rb'))
pca_all = pickle.load(open(os.path.join(DATA_DIR, 'PCA_model_rp_sa_10traits.pkl'), 'rb'))
print('explained variance ratio:', np.round(pca_all.explained_variance_ratio_, 4))

# 3-D arrow and annotation helpers

In [ ]:
# 3-D arrow and annotation helpers, from https://gist.github.com/WetHat/1d6cd0f7309535311a539b42cccca89c
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d.proj3d import proj_transform
from mpl_toolkits.mplot3d.axes3d import Axes3D

from matplotlib.text import Annotation


class Annotation3D(Annotation):

    def __init__(self, text, xyz, *args, **kwargs):
        super().__init__(text, xy=(0, 0), *args, **kwargs)
        self._xyz = xyz

    def draw(self, renderer):
        x2, y2, z2 = proj_transform(*self._xyz, self.axes.M)
        self.xy = (x2, y2)
        super().draw(renderer)


def _annotate3D(ax, text, xyz, *args, **kwargs):
    '''Add anotation `text` to an `Axes3d` instance.'''

    annotation = Annotation3D(text, xyz, *args, **kwargs)
    ax.add_artist(annotation)


setattr(Axes3D, 'annotate3D', _annotate3D)


class Arrow3D(FancyArrowPatch):

    def __init__(self, x, y, z, dx, dy, dz, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._xyz = (x, y, z)
        self._dxdydz = (dx, dy, dz)

    def draw(self, renderer):
        x1, y1, z1 = self._xyz
        dx, dy, dz = self._dxdydz
        x2, y2, z2 = (x1 + dx, y1 + dy, z1 + dz)

        xs, ys, zs = proj_transform((x1, x2), (y1, y2), (z1, z2), self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        super().draw(renderer)

    def do_3d_projection(self, renderer=None):
        x1, y1, z1 = self._xyz
        dx, dy, dz = self._dxdydz
        x2, y2, z2 = (x1 + dx, y1 + dy, z1 + dz)

        xs, ys, zs = proj_transform((x1, x2), (y1, y2), (z1, z2), self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))

        return np.min(zs)


def _arrow3D(ax, x, y, z, dx, dy, dz, *args, **kwargs):
    '''Add an 3d arrow to an `Axes3D` instance.'''

    arrow = Arrow3D(x, y, z, dx, dy, dz, *args, **kwargs)
    ax.add_artist(arrow)


setattr(Axes3D, 'arrow3D', _arrow3D)

# Fig. 2a - 3-D trait space coloured by NLCD class

In [ ]:
sns.set_style('ticks')
sns.set_context('paper')

# project the sample onto the published axes: log -> standardise -> PCA
# (pca_all.components_ already carry the sign flip applied in notebook 01)
X = np.log(df[trait_list])
X_scale = scaler_all.transform(X)
X_scale_reduced = pca_all.transform(X_scale)
X_scale_reduced = pd.DataFrame(X_scale_reduced, index=df.index, columns=['PC1', 'PC2', 'PC3'])

In [ ]:
norm = matplotlib.colors.Normalize(vmin=0, vmax=6)
rgba = cm.gist_ncar([norm(0), norm(1), norm(2), norm(3), norm(4), norm(5), norm(6)])

# rgba to list of hex colours, one per NLCD class
color_list = []
for i in range(rgba.shape[0]):
    color_list.append(matplotlib.colors.rgb2hex(rgba[i, :3]))

color_list[2] = '#006E00'
color_list[6] = '#7F7F7F'

fig = plt.figure(figsize=(5/1.15, 5/1.15))
ax = fig.add_subplot(111, projection='3d')
# scatter in 3-D, one colour per NLCD class
X_scale_reduced['nlcd'] = df['nlcd']
for color, nlcd in zip(color_list, ['Deciduous Forest', 'Mixed Forest', 'Evergreen Forest', 'Shrub/Scrub',
                                    'Grassland/Herbaceous', 'Woody Wetlands']):
    ax.scatter(X_scale_reduced.query('nlcd==@nlcd')['PC1'], X_scale_reduced.query('nlcd==@nlcd')['PC2'],
               X_scale_reduced.query('nlcd==@nlcd')['PC3'], linewidth=0, s=1.5,
               color=color,
               **{'edgecolor': 'none', 'alpha': 0.3})

# trait loadings as 3-D arrows
for i in range(len(trait_list)):
    enlarge = 10
    enlarge_z = 7
    arrow_color = '#05445E'
    ax.arrow3D(0, 0, 0, pca_all.components_[0, i] * enlarge, pca_all.components_[1, i] * enlarge,
               pca_all.components_[2, i] * enlarge_z,
               linewidth=1.2,
               fc=arrow_color, ec=arrow_color,
               mutation_scale=10,
               arrowstyle="-|>",
               linestyle='dashed', zorder=100)
    # position of the label relative to the arrow tip
    xytext = (0, 0)
    if pca_all.components_[0, i] > 0:
        ha = 'left'
    else:
        ha = 'right'
    if pca_all.components_[1, i] > 0:
        va = 'bottom'
    else:
        va = 'top'
    if trait_list[i] in ['Canopy Height']:
        va = 'center'
        xytext = (0, 8)
    if trait_list[i] in ['Phenolics','Lignin']:
        va = 'top'
        xytext = (0, -5)
    if trait_list[i] in ['EWT', ]:
        va = 'top'
        ha = 'center'
    if trait_list[i] in ['Carbon', ]:
        va = 'center'
    if trait_list[i] in ['SLA','Chlorophyll a + b']:
        ha = 'center'
    ax.annotate3D(trait_list[i], (pca_all.components_[0, i] * enlarge, pca_all.components_[1, i] * enlarge,
                                  pca_all.components_[2, i] * enlarge_z),
                  bbox=dict(boxstyle="round", fc="lightyellow", ec='k'),
                  va=va, ha=ha,
                  xytext=xytext, textcoords='offset points', zorder=100)

# viewing angle
ax.view_init(elev=33, azim=-70)
ax.set_box_aspect((4, 4, 3.5), zoom=1.)

ax.tick_params(axis='x', which='major', pad=-4)
ax.xaxis.labelpad = -6
ax.tick_params(axis='y', which='major', pad=-3)
ax.yaxis.labelpad = -6
ax.tick_params(axis='z', which='major', pad=-3)
ax.zaxis.labelpad = -8

# pane colours
ax.xaxis.pane.set_facecolor('1')
ax.yaxis.pane.set_facecolor('0.98')
ax.zaxis.pane.set_facecolor('0.98')
ax.set_xlabel("PC{} ({}%)".format(1, round(pca_all.explained_variance_ratio_[0] * 100, 1)))
ax.set_ylabel('PC{} ({}%)'.format(2, round(pca_all.explained_variance_ratio_[1] * 100, 1)))
ax.set_ylim(-4, 6)
ax.set_zlabel('PC{} ({}%)'.format(3, round(pca_all.explained_variance_ratio_[2] * 100, 1)))
ax.set_zlim(-6, 6)
fig.tight_layout()
plt.show()
fig.savefig(os.path.join(OUT_DIR, 'fig2a_trait_space_3d.png'), dpi=1000, bbox_inches='tight')
plt.close(fig)